# Práctica



## Preparación previa

### Importaciones

In [1]:
import altair as alt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### Instalación de Altair

Para la instalación de las librerías necesarias, he creado un entorno en conda llamado `unedviz`, utilizando el comando:

```
conda create --name unedviz python=3.12
conda activate unedviz
```

Después, he procedido a instalar `altair` haciendo:
```
conda install -c conda-forge altair-all
```

Y he creado el kernel para jupyter:
```
python -m ipykernel install --user --name=unedviz --display-name "Python (unedviz)"
```

Finalmente, para comprobar la correcta instalación, he ejecutado el código de muestra de la web de Altair, instalando previamente los datasets de muestra con el comando:
```
pip install vega_datasets
```

In [2]:
# load a sample dataset as a pandas DataFrame
from vega_datasets import data
cars = data.cars()

# make the chart
alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
).interactive()

alt.Chart(...)

### Carga del dataset

Los datos se encuentran en la carpeta `./CSV Files`. Estos son los archivos que tenemos:

- **`.\CSV Files\The UNSW-NB15 description.pdf`**: Archivo que describe el dataset. Indica que el conjunto de datos UNSW-NB15 fue generado en el laboratorio Cyber Range de UNSW Canberra, combinando tráfico normal real y ataques sintéticos, y contiene más de 2.5 millones de registros con 49 características, distribuidos en archivos CSV y clasificados por tipos de ataques como DoS, Exploits, y Malware. Se utilizaron herramientas como Tcpdump, Argus y Bro-IDS, y se incluyen particiones para entrenamiento (175,341 registros) y prueba (82,332 registros).
- **`.\CSV Files\NUSW-NB15_features.csv`**: Incluye una lista de 49 características, con su nombre, tipo de datos (`integer`, `nominal`, etc.) y descripción.
- **Registros de datos**: contienen los registros verdaderos de datos de tráfico real y ataques sintéticos combinados, y sus etiquetas.
    - **`.\CSV Files\UNSW-NB15_1.csv`**: Primer CSV. Contiene 49 columnas de datos, y cada columna corresponde a una de las características listadas en el archivo `NUSW-NB15_features.csv`. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_2.csv`**: Segundo CSV. La estructura es igual al archivo anterior. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_3.csv`**: Tercer archivo, contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_4.csv`**: Cuarto archivo, contiene 440 044 registros.
- **`.\CSV Files\NUSW-NB15_GT.csv`**: Registra una lista de los eventos de los ataques. Es el *ground truth*, contiene la verdad conocida o etiquetas reales de los datos: es decir, indica con certeza qué tipo específico de ataque es. Actúa como un archivo de referencia independiente, y con una estructura más limpia, usada para validar/relacionar los datos de otra manera. Para cada ataque, incluye su hora de inicio y hora final, la categoría de ataque (p.ej. *Backdoor*, *Exploit*, etc.), subcategoría, protocolo utilizado, IP y puerto origen, IP y puerto destino, nombre del ataque y su referencia (CVE, BID, etc.).
- **`.\CSV Files\UNSW-NB15_LIST_EVENTS.csv`**: Resumen agregado de los eventos. Contiene el número de eventos totales de cada categoría y subcategoría de ataque (datos agregados).
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_training-set.csv`**: Se trata de una partición de los archivos de datos con 175 341 registros. Sin embargo, el objetivo de este training set en concreto es utilizarlo para el entrenamiento de modelos de Machine Learning.
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_testing-set.csv`**: Igual que el training set, se trata de una partición de los archivos de datos con 82 332 registros. El objetivo de este testing set es utilizarlo para validar el entrenamiento de modelos de Machine Learning.


In [19]:
# Cargar nombres de columnas
features_path = r'.\CSV Files\NUSW-NB15_features.csv'
features_df = pd.read_csv(features_path, encoding='latin1')
features_df.columns = features_df.columns.str.strip()

# Crear diccionario de conversión de tipos
type_mapping = {
    'Float': 'float32',
    'Integer': 'Int64',
    'integer': 'Int64',
    'Binary': 'Int8',
    'binary': 'Int8',
    'nominal': 'category',
    'Timestamp': 'str'
}

# Crear diccionario que asocie cada nombre de la columna (clave) con su tipo de datos (valor)
dtype_dict = {}
for index, row in features_df.iterrows():
    column_name = row['Name']
    column_type = type_mapping.get(row['Type'], 'object')
    dtype_dict[column_name] = column_type


In [20]:
# Archivos de datos sin encabezado
data_files = [
    r'.\CSV Files\UNSW-NB15_1.csv',
    r'.\CSV Files\UNSW-NB15_2.csv',
    r'.\CSV Files\UNSW-NB15_3.csv',
    r'.\CSV Files\UNSW-NB15_4.csv'
]

# Leer y concatenar todos los archivos
dataframes = []
for file in data_files:
    # Leer archivo sin asignar tipos
    df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')
    dataframes.append(df)

# Concatenar los DataFrames
full_data = pd.concat(dataframes, ignore_index=True)

C:\Users\maial\AppData\Local\Temp\ipykernel_65572\1844327936.py:13: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')
C:\Users\maial\AppData\Local\Temp\ipykernel_65572\1844327936.py:13: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')


Carga de los datos realizado con éxito.

## Hipótesis

#### Significado de las columnas

Cada entrada del dataset representa un flujo de red (también llamado *network flow* o *network connection*). Se corresponde a una única conexión entre un dispositivo de origen y uno de destino, durante un intervalo de tiempo, con información sobre lo que ocurrió en esa conexión.

Las columnas de cada entrada del dataset UNSW-NB15 son:

| **Nombre de columna**       | **Descripción**                                                                                  |
|-----------------------------|-------------------------------------------------------------------------------------------------------------|
| `srcip`                     | Dirección IP de origen                                                                                      |
| `sport`                     | Puerto de origen                                                                                             |
| `dstip`                     | Dirección IP de destino                                                                                      |
| `dsport`                    | Puerto de destino                                                                                            |
| `proto`                     | Protocolo de la transacción                                                                                  |
| `state`                     | Estado de la conexión y protocolo asociado (p. ej., ACC, FIN, RST, etc.)                                     |
| `dur`                       | Duración total del registro                                                                                  |
| `sbytes`                    | Bytes transferidos desde el origen al destino                                                                |
| `dbytes`                    | Bytes transferidos desde el destino al origen                                                                |
| `sttl`                      | Valor TTL (time to live) de origen a destino                                                                 |
| `dttl`                      | Valor TTL de destino a origen                                                                                |
| `sloss`                     | Paquetes reenviados o perdidos desde el origen                                                               |
| `dloss`                     | Paquetes reenviados o perdidos desde el destino                                                              |
| `service`                   | Servicio usado (http, ftp, smtp, ssh, dns, etc.)                                                             |
| `Sload`                     | Bits por segundo enviados por el origen                                                                      |
| `Dload`                     | Bits por segundo recibidos en el destino                                                                     |
| `Spkts`                     | Número de paquetes enviados del origen al destino                                                            |
| `Dpkts`                     | Número de paquetes enviados del destino al origen                                                            |
| `swin`                      | Valor de ventana de anuncios TCP del origen                                                                  |
| `dwin`                      | Valor de ventana de anuncios TCP del destino                                                                 |
| `stcpb`                     | Número base de secuencia TCP del origen                                                                      |
| `dtcpb`                     | Número base de secuencia TCP del destino                                                                     |
| `smeansz`                   | Tamaño medio de los paquetes enviados por el origen                                                          |
| `dmeansz`                   | Tamaño medio de los paquetes enviados por el destino                                                         |
| `trans_depth`              | Profundidad de conexión en solicitudes/respuestas HTTP                                                       |
| `res_bdy_len`              | Tamaño del contenido real sin comprimir transferido desde el servidor HTTP                                   |
| `Sjit`                      | Jitter (variación en la latencia) del origen en milisegundos                                                 |
| `Djit`                      | Jitter del destino en milisegundos                                                                           |
| `Stime`                     | Tiempo de inicio del registro                                                                                |
| `Ltime`                     | Tiempo final del registro                                                                                     |
| `Sintpkt`                   | Tiempo entre paquetes del origen                                                                             |
| `Dintpkt`                   | Tiempo entre paquetes del destino                                                                            |
| `tcprtt`                    | Tiempo de ida y vuelta para establecer conexión TCP (synack + ackdat)                                        |
| `synack`                    | Tiempo entre SYN y SYN-ACK                                                                                   |
| `ackdat`                    | Tiempo entre SYN-ACK y ACK                                                                                   |
| `is_sm_ips_ports`          | Vale 1 si la IP y puertos de origen/destino son iguales; si no, vale 0                                       |
| `ct_state_ttl`              | Número de veces que aparece un estado particular en función de los TTL del origen y destino                  |
| `ct_flw_http_mthd`          | Número de flujos que usan métodos HTTP como GET o POST                                                       |
| `is_ftp_login`              | Vale 1 si hay acceso FTP con usuario y contraseña; si no, vale 0                                             |
| `ct_ftp_cmd`                | Número de flujos que tienen comandos FTP                                                                     |
| `ct_srv_src`                | Número de conexiones que comparten el mismo servicio y dirección IP de origen en los últimos 100 registros    |
| `ct_srv_dst`                | Número de conexiones que comparten el mismo servicio y dirección IP de destino en los últimos 100 registros   |
| `ct_dst_ltm`                | Número de conexiones con la misma dirección IP de destino en los últimos 100 registros                       |
| `ct_src_ltm`                | Número de conexiones con la misma dirección IP de origen en los últimos 100 registros                        |
| `ct_src_dport_ltm`          | Número de conexiones con la misma IP de origen y puerto de destino en los últimos 100 registros              |
| `ct_dst_sport_ltm`          | Número de conexiones con la misma IP de destino y puerto de origen en los últimos 100 registros              |
| `ct_dst_src_ltm`            | Número de conexiones entre una IP de origen y una de destino específicas en los últimos 100 registros         |
| `attack_cat`                | Categoría del ataque (Fuzzers, DoS, Shellcode, Worms, etc.)                                                  |
| `Label`                     | Etiqueta binaria: 0 = tráfico normal, 1 = tráfico malicioso                                                  |



### Amenazas y su impacto en los datos del UNSW-NB15


| **Nombre amenaza** | **Descripción** | **Alteración esperada en los datos** |
|--------------------|------------------|----------------------------------------|
| **Fuzzers** | Ataques que buscan hacer fallar la red generando paquetes con datos aleatorios. | - Incremento en `Spkts` y `Dpkts` por el envío masivo de paquetes.<br>- Aumento de `sloss` y `dloss` debido a paquetes mal formados que se pierden o descartan.<br>- `smeansz` y `dmeansz` con tamaños de paquete inusuales.<br>- Alta variabilidad en `Sjit`, `Djit`, `Sintpkt`, `Dintpkt` por comportamiento errático. |
| **Analysis** | Ataques de escaneo de puertos, spam e intentos de modificar páginas HTML. | - Elevado número de conexiones cortas con `dur` muy bajo.<br>- Posible aumento en `ct_flw_http_mthd` por múltiples métodos HTTP (GET, POST).<br>- `Sload` y `Dload` bajos pero con muchas instancias.<br>- Cambios en `state`, apareciendo muchos estados como `REQ`, `RST`, `FIN`.<br>- Incremento en `ct_srv_dst`, `ct_dst_ltm` por repetición de destino. |
| **Backdoors** | Acceso no autorizado eludiendo mecanismos de seguridad de forma sigilosa. | - `dur` largo con `Sload` y `Dload` moderados o altos (canales ocultos de datos).<br>- Actividad anormal en puertos comunes (`sport`, `dsport` inusuales).<br>- `is_sm_ips_ports` puede ser 1 (si se intenta camuflar tráfico como local).<br>- Uso de servicios poco comunes en `service`.<br>- `state` puede mostrar conexiones abiertas de largo plazo (`CON`, `INT`). |
| **DoS** | Inundación de peticiones para agotar recursos de un sistema. | - Aumento **brutal** en `Spkts`, `Dpkts`, `sbytes`, `dbytes`.<br>- `Sload` y `Dload` altísimos por congestión de red.<br>- `dur` muy bajo (peticiones rápidas, masivas).<br>- `sloss` y `dloss` altos debido a la saturación.<br>- `ct_dst_ltm`, `ct_srv_dst` elevados (muchas conexiones al mismo destino). |
| **Exploits** | Ataques que aprovechan vulnerabilidades para obtener acceso o privilegios. | - `dur` moderado-alto con `sbytes` o `dbytes` elevados en pocos registros.<br>- `stcpb`, `dtcpb` pueden mostrar patrones inusuales si se manipulan secuencias.<br>- `state` puede mostrar transiciones anómalas.<br>- `ct_ftp_cmd`, `ct_flw_http_mthd` elevados si se explotan protocolos específicos.<br>- Picos en `tcprtt`, `synack`, `ackdat` si se manipula el handshake TCP. |
| **Generic** | Ataques genéricos a cifrados de bloques, basados en conocimiento de su estructura. | - `dur` muy corto con patrones repetitivos.<br>- `smeansz`, `dmeansz` constantes o sospechosamente iguales (bloques estándar).<br>- `Sload`, `Dload` no necesariamente altos, pero frecuentes.<br>- Posible repetición de IPs o puertos (`ct_src_ltm`, `ct_dst_ltm`, etc.). |
| **Reconnaissance** | Recopilación de información (tipo escaneo) con herramientas como Strikes. | - Muchas conexiones muy cortas (`dur` muy bajo).<br>- `Spkts`, `Dpkts` bajos pero con alta frecuencia.<br>- `state` = `REQ`, `RST`, `INT` comúnmente.<br>- `ct_src_ltm`, `ct_srv_dst`, `ct_dst_ltm` muy altos por intentos a muchos destinos.<br>- Uso de puertos inusuales para escanear (`sport`, `dsport`). |
| **Shellcode** | Fragmentos de código malicioso que explotan una vulnerabilidad dentro de un programa. | - Tamaños de paquetes (`smeansz`, `dmeansz`) muy pequeños o precisos (para ejecutar payloads).<br>- `sbytes`, `dbytes` bajos en una conexión sospechosamente efectiva.<br>- `tcprtt`, `synack`, `ackdat` podrían mostrar valores alterados.<br>- `service` puede ser HTTP o FTP si se intenta inyectar vía tráfico normal.<br>- `ct_ftp_cmd` o `ct_flw_http_mthd` pueden mostrar comandos inusuales. |
| **Worms** | Malware que se auto-replica e intenta propagarse automáticamente. | - Muchísimas conexiones en poco tiempo desde la misma IP → `ct_src_ltm`, `ct_src_dport_ltm` muy altos.<br>- `Sload`, `Dload`, `Spkts`, `Dpkts` aumentados conforme se replica.<br>- `res_bdy_len` puede crecer si se transfiere el malware como archivo.<br>- Cambios en `state` mostrando múltiples inicios/terminaciones (`REQ`, `FIN`, `RST`).<br>- `is_sm_ips_ports` puede ser 1 si intenta propagarse localmente. |


## Preparación de los datos (preprocesado)

### Limpieza y asignación de tipos

In [16]:
# Función para convertir un valor en hexadecimal a decimal y ' ' o '-' a -1
def convert_to_int(value):
    try:
        str_value = str(value).strip()
    
        if str_value.startswith('0x'): # Hexadecimal
            return int(str_value, 16)
        elif str_value in ['', '-']:
            return None
        else:
            return int(str_value)
    except ValueError:
        return value

La limpieza de los valores enteros se caracteriza sobretodo por pasar los valores hexadecimales a decimales, y por reemplazar valores como `['', '-']` con `None`.

In [21]:
# Aplicar la conversión a enteros
for column, dtype in dtype_dict.items():
    if dtype == 'Int64':
        full_data[column] = full_data[column].apply(convert_to_int)


# Convertir las columnas al tipo correcto según el dtype_dict
for column, dtype in dtype_dict.items():
    full_data[column] = full_data[column].astype(dtype)

In [23]:
# Mostrar info general
print(full_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Dtype   
---  ------            -----   
 0   srcip             category
 1   sport             Int64   
 2   dstip             category
 3   dsport            Int64   
 4   proto             category
 5   state             category
 6   dur               float32 
 7   sbytes            Int64   
 8   dbytes            Int64   
 9   sttl              Int64   
 10  dttl              Int64   
 11  sloss             Int64   
 12  dloss             Int64   
 13  service           category
 14  Sload             float32 
 15  Dload             float32 
 16  Spkts             Int64   
 17  Dpkts             Int64   
 18  swin              Int64   
 19  dwin              Int64   
 20  stcpb             Int64   
 21  dtcpb             Int64   
 22  smeansz           Int64   
 23  dmeansz           Int64   
 24  trans_depth       Int64   
 25  res_bdy_len       

### Análisis Exploratorio de los Datos (EDA)

In [24]:
# Valores nulos
full_data.isnull().sum()

srcip                     0
sport                     2
dstip                     0
dsport                    7
proto                     0
state                     0
dur                       0
sbytes                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
Sload                     0
Dload                     0
Spkts                     0
Dpkts                     0
swin                      0
dwin                      0
stcpb                     0
dtcpb                     0
smeansz                   0
dmeansz                   0
trans_depth               0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
Sintpkt                   0
Dintpkt                   0
tcprtt                    0
synack                    0
ackdat                    0
is_sm_ips_ports     

In [25]:
full_data.duplicated().sum()

np.int64(480633)

#### Análisis de las Variables Numéricas

In [26]:
# Ver estadística descriptiva
full_data.describe()

,sport,dsport,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,...,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,Label
count,2540045.0,2540040.0,2.540047e+06,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0,2.540047e+06,...,1110168.0,1110168.0,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0,2540047.0
mean,30534.498578,11664.183432,6.587917e-01,4339.600064,36427.593728,62.781975,30.76681,5.163921,16.329444,3.695645e+07,...,0.039699,0.047036,9.206988,8.988958,6.439103,6.900986,4.642139,3.592729,6.845886,0.126487
std,20442.146335,478617.315153,1.392493e+01,56405.994681,161096.035542,74.622765,42.850888,22.517075,56.594744,1.186043e+08,...,0.199659,0.276626,10.836755,10.822489,8.162034,8.205062,8.477579,6.174445,11.258282,0.332398
min,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
25%,11226.0,53.0,1.037000e-03,200.0,178.0,31.0,29.0,0.0,0.0,1.353963e+05,...,0.0,0.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,0.0
50%,31687.0,80.0,1.586100e-02,1470.0,1820.0,31.0,29.0,3.0,4.0,5.893038e+05,...,0.0,0.0,5.0,5.0,3.0,4.0,1.0,1.0,2.0,0.0
75%,47439.0,14987.0,2.145545e-01,3182.0,14894.0,31.0,29.0,7.0,14.0,2.039923e+06,...,0.0,0.0,10.0,10.0,6.0,7.0,2.0,1.0,5.0,0.0
max,65535.0,538989345.0,8.786638e+03,14355774.0,14657531.0,255.0,254.0,5319.0,5507.0,5.988000e+09,...,4.0,8.0,67.0,67.0,67.0,67.0,67.0,60.0,67.0,1.0


Ahora veremos la distribución de estas variables numéricas en formato de gráfico, para intentar obtener más detalle:

In [32]:
alt.data_transformers.enable("vegafusion")
# Filtramos solo las columnas numéricas
numeric_cols = full_data.select_dtypes(include=['number'])

# Recorrer cada columna numérica para mostrar distribución
for col in numeric_cols.columns:
    print(f"Distribución de la columna {col}:")
        
    # Histograma
    hist = alt.Chart(full_data).mark_bar().encode(
        alt.X(f'{col}:Q',bin=alt.Bin(maxbins=300)),
        alt.Y('count():Q')
    ).properties(
        title=f'Histograma de {col}',
        width=600,
        height=400
    ).interactive()
    
    # Mostrar ambos gráficos
    hist.show()

Distribución de la columna sport:


alt.Chart(...)

Distribución de la columna dsport:


alt.Chart(...)

Distribución de la columna dur:


...

alt.Chart(...)

Distribución de la columna ct_dst_src_ltm:


alt.Chart(...)

Distribución de la columna Label:


alt.Chart(...)

En algunos casos parece que la distribución no tiene sentido, pero lo que realmente pasa es que tenemos outliers que dificultan la correcta visualización en el plot.

#### Análisis de las Variables Categóricas

In [34]:
# Método para contar y calcular el % de las categorías
def count_and_percent(df, column_name):
    count_data = df[column_name].value_counts().reset_index()
    count_data.columns = [column_name, 'count']
    count_data['%'] = (count_data['count'] / count_data['count'].sum()) * 100
    return count_data

In [35]:
# Aplicar el método a todas las columnas categóricas
categorical_columns = full_data.select_dtypes(include=['category']).columns

# Crear un diccionario o una lista para almacenar los resultados
results = {}

for col in categorical_columns:
    results[col] = count_and_percent(df, col)

# Mostrar resultados (por ejemplo, para la columna 'neighbourhood_group')
for col, result in results.items():
    print(f"Distribución para {col}:")
    print(result)
    print("\n")

Distribución para srcip:
             srcip  count         %
0   149.171.126.14  41037  9.325658
1     175.45.176.1  40538  9.212261
2     175.45.176.0  38614  8.775032
3   149.171.126.10  30356  6.898401
4       59.166.0.1  27391  6.224605
5       59.166.0.4  27216  6.184836
6       59.166.0.5  27164  6.173019
7       59.166.0.0  27111  6.160975
8       59.166.0.2  27050  6.147113
9       59.166.0.3  26900  6.113025
10      59.166.0.9  26398  5.998946
11      59.166.0.7  26055  5.920999
12      59.166.0.8  26024  5.913954
13      59.166.0.6  25590  5.815328
14    175.45.176.2  11805  2.682686
15    175.45.176.3   7981  1.813682
16     10.40.182.6    964  0.219069
17      10.40.85.1    415  0.094309
18     10.40.182.1    403  0.091582
19     10.40.85.10    225  0.051131
20     10.40.85.30    220  0.049995
21     10.40.182.3    219  0.049768
22     10.40.170.2    216  0.049086
23  149.171.126.13    106  0.024089
24  149.171.126.19      7  0.001591
25  149.171.126.18      7  0.001591
26 